# NN Training to Substitute LUT

## Setup
Set-up Keras in configuration "torch" to use PyTorch.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

## LUT Upload
Upload Lut object from a `.nc` file.

In [ ]:
from src.lut.lut import Lut

LUT_PATH = "models/luts/dry_run/dry_run_lut_p2_W_0.15.nc"

my_lut = Lut.load(LUT_PATH)
my_lut.print_info()

## LUT Visualization
Heat-map to visualize the intensity of the muscle.
Insert negative values for `ANGLE_REQ` or `TAU_REQ` if you wanto to analize the general case.

In [ ]:
THETA_REQ = 45.0
THETA_DOT_REQ = 0.0
TAU_NORM_REQ = 0.5

CONSIDERED_W = 0.5

SAVE_PLOTS_DIR = None

# choose plot to be shown
SHOW_TAU_0 = False
SHOW_HEATMAPS_K = False
SHOW_TAU_MIN_MAX = False
SHOW_TAU_WITH_TAU_MIN_MAX = False
SHOW_HEATMAPS_ACTIVATIONS = False
SHOW_COST = False
SHOW_COST_ALL_W = False
SHOW_HEATMAP_COST = False
SHOW_COST_CURVES = True
COMPARE_P = [2, 3] # p values compared at CONSIDERED_W, no LUT needed


def plot_path(name: str):
    return os.path.join(SAVE_PLOTS_DIR, name) if SAVE_PLOTS_DIR else None

def wanted(show: bool) -> bool:
    return show or SAVE_PLOTS_DIR is not None

if wanted(SHOW_TAU_0):
    my_lut.plot_tau_0(save_path=plot_path("tau_0.png"),
                      show=SHOW_TAU_0)
if wanted(SHOW_HEATMAPS_K):
    my_lut.plot_heatmaps_k(theta_req=THETA_REQ,
                           theta_dot_req=THETA_DOT_REQ,
                           save_path=plot_path("hmps_k.png"),
                           show=SHOW_HEATMAPS_K)
if wanted(SHOW_TAU_MIN_MAX):
    my_lut.plot_tau_min_max(velocity=THETA_DOT_REQ,
                            save_path=plot_path(f"tau_min_max_v_{THETA_DOT_REQ:g}.png"),
                            show=SHOW_TAU_MIN_MAX)
if wanted(SHOW_TAU_WITH_TAU_MIN_MAX):
    my_lut.plot_tau_with_tau_min_max(velocity=THETA_DOT_REQ,
                                    save_path=plot_path(f"tau_v_{THETA_DOT_REQ:g}.png"),
                                    show=SHOW_TAU_WITH_TAU_MIN_MAX)
if wanted(SHOW_HEATMAPS_ACTIVATIONS):
    my_lut.plot_heatmaps_activations(w=CONSIDERED_W,
                                    velocity=THETA_DOT_REQ,
                                    theta_req=THETA_REQ,
                                    tau_norm_req=TAU_NORM_REQ,
                                    save_path=plot_path(f"hmps_acts_{CONSIDERED_W:g}.png"),
                                    show=SHOW_HEATMAPS_ACTIVATIONS)
if wanted(SHOW_COST):
    my_lut.plot_cost(w=CONSIDERED_W,
                     save_path=plot_path(f"cost_{CONSIDERED_W:g}.png"),
                     show=SHOW_COST)
if wanted(SHOW_COST_ALL_W):
    my_lut.plot_cost_all_w(save_path=plot_path("cost_all_w.png"),
                           show=SHOW_COST_ALL_W)
if wanted(SHOW_HEATMAP_COST):
    my_lut.plot_heatmap_cost(w=CONSIDERED_W,
                             velocity=THETA_DOT_REQ,
                             theta_req=THETA_REQ,
                             tau_norm_req=TAU_NORM_REQ,
                             save_path=plot_path(f"hmp_cost_{CONSIDERED_W:g}.png"),
                             show=SHOW_HEATMAP_COST)
if wanted(SHOW_COST_CURVES):
    Lut.plot_cost_curves([(p, CONSIDERED_W) for p in COMPARE_P],
                         save_path=plot_path(f"cost_p_compare_W_{CONSIDERED_W:g}.png"),
                         show=SHOW_COST_CURVES)

## NN Setup
We need to define the number of layers and of neurons of the Neural Network.
The input is $x_{in}=[W,\theta, \dot\theta, \tau]^T$ and output is $u=[u(1),\dots, u(6)]^T$.

In [ ]:
from src.nn.nn_generator import NnGenerator

NN_NAME = "model_W_TRY"
NUM_HIDDEN_LAYERS = 1
HIDDEN_UNITS = 2
HIDDEN_FUNCTION = "relu"
OUTPUT_FUNCTION = "linear"
SEED = 42

HOME_DIR = "models/nn/W_all"

my_nn_generator = NnGenerator(my_lut)

my_nn_generator.define_structure_nn(num_hidden_layers=NUM_HIDDEN_LAYERS,
                                    hidden_units=HIDDEN_UNITS,
                                    hidden_function=HIDDEN_FUNCTION,
                                    output_function=OUTPUT_FUNCTION,
                                    seed=SEED)
my_nn_generator.model.summary()

## NN Training

shuffle=True,       # Rimescola i batch a ogni epoca per evitare bias di ordinamento

In [ ]:
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

DIR_NN = os.path.join(HOME_DIR, NN_NAME)
SAVE_PATH_PLOTS = os.path.join(DIR_NN, "plots")

my_nn_generator.train_nn(nn_name=NN_NAME,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         learning_rate=LEARNING_RATE)
my_nn = my_nn_generator.nn
my_nn.save(DIR_NN, lut=my_lut)

## Plots & Save Plots

In [ ]:
THETA_REQ = None #45.0
TAU_NORM_REQ = None #0.5
THETA_DOT_REQ = 0.0

CONSIDERED_W = 0.5

SAVE_PLOTS_DIR = SAVE_PATH_PLOTS # None -> show only, nothing saved

# choose plot to be shown
SHOW_INFO = False
SHOW_STRUCTURE = False
SHOW_LOSS = False
SHOW_LEARNING_RATE = False
SHOW_HEATMAPS_ACTIVATIONS = False
SHOW_HEATMAPS_ERROR = False
SHOW_HEATMAP_TORQUE_ERROR = False
SHOW_LUT_COMPARISON = False
SHOW_PARITY = True


def plot_path(name: str):
    return os.path.join(SAVE_PLOTS_DIR, name) if SAVE_PLOTS_DIR else None

def wanted(show: bool) -> bool:
    return show or SAVE_PLOTS_DIR is not None

tag = f"W_{CONSIDERED_W:g}_v_{THETA_DOT_REQ:g}"

if SHOW_INFO:
    my_nn.print_info()
if wanted(SHOW_STRUCTURE):
    my_nn.plot_structure(save_path=plot_path(f"{NN_NAME}_graph.png"),show=SHOW_STRUCTURE)
if wanted(SHOW_LOSS):
    my_nn.plot_loss(save_path=plot_path(f"{NN_NAME}_loss.png"),show=SHOW_LOSS)
if wanted(SHOW_LEARNING_RATE):
    my_nn.plot_learning_rate(save_path=plot_path(f"{NN_NAME}_learning_rate.png"),show=SHOW_LEARNING_RATE)
if wanted(SHOW_HEATMAPS_ACTIVATIONS):
    my_nn.plot_heatmaps_activations(my_lut, w=CONSIDERED_W, velocity=THETA_DOT_REQ, theta_req=THETA_REQ, tau_norm_req=TAU_NORM_REQ,save_path=plot_path(f"{NN_NAME}_hmps_u_{tag}.png"),show=SHOW_HEATMAPS_ACTIVATIONS)
if wanted(SHOW_HEATMAPS_ERROR):
    my_nn.plot_heatmaps_error(my_lut, w=CONSIDERED_W, velocity=THETA_DOT_REQ, theta_req=THETA_REQ, tau_norm_req=TAU_NORM_REQ,save_path=plot_path(f"{NN_NAME}_hmps_error_{tag}.png"), show=SHOW_HEATMAPS_ERROR)
if wanted(SHOW_HEATMAP_TORQUE_ERROR):
    my_nn.plot_heatmap_torque_error(my_lut, w=CONSIDERED_W, velocity=THETA_DOT_REQ, theta_req=THETA_REQ, tau_norm_req=TAU_NORM_REQ, save_path=plot_path(f"{NN_NAME}_hmp_torque_error_{tag}.png"), show=SHOW_HEATMAP_TORQUE_ERROR)
if wanted(SHOW_LUT_COMPARISON):
    my_lut.plot_heatmaps_activations(w=CONSIDERED_W, velocity=THETA_DOT_REQ, theta_req=THETA_REQ, tau_norm_req=TAU_NORM_REQ,save_path=plot_path(f"lut_hmps_acts_{tag}.png"), show=SHOW_LUT_COMPARISON)
if wanted(SHOW_PARITY):
    my_nn.plot_parity(my_lut, save_path=plot_path(f"{NN_NAME}_parity.png"), show=SHOW_PARITY)